# Reproduce LMLM paper — preprocessing walkthrough

This notebook explains and **executes** the preprocessing pipeline currently implemented in MalWeave. Its goal is to transform PE files stored in ZIP archives into the paper's three representations:

- **EXE**: bytes from PE sections marked as executable or containing code.
- **DIS**: the instruction stream produced by Ghidra, excluding signatures, addresses, and raw instruction bytes.
- **DEC**: C-like code produced by the Ghidra decompiler.

The manifest is only an audit trail explaining why each sample was kept or rejected. The actual data for downstream stages is written to `raw/`, `exe/`, `dis/`, and `dec/`.

## 1. Reproduction flow

```text
ZIP archives containing PE files
    │
    ├── not PE32 x86 ──────────────────────────> reject
    │
    ├── DIE detects packer/protector/crypter --> reject
    │
    └── accepted sample
          ├── full PE --------------------------> raw/
          ├── executable-section bytes --------> exe/
          └── Ghidra headless
                ├── instruction-only stream ---> dis/
                └── decompiled C-like code ----> dec/
```

The filter order follows Section II of the paper. Architecture filtering must run before Ghidra because Ghidra is configured for `x86:LE:32`.

## 2. Environment setup

- Never execute any binary directly. The pipeline only reads PE files and performs static analysis in Docker.
- Use the project's virtual environment with `lief`, `pefile`, and `tqdm`.
- Docker is required for Detect-It-Easy and Ghidra. Their images are built automatically on the first run.
- Output is stored in the ignored directory `data/ranDS/processed/reproduction/`. Never commit malware or derived data to Git.

In [19]:
from dataclasses import asdict, dataclass
import json
from pathlib import Path
from pprint import pprint
import subprocess
import sys
import zipfile

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from malweave.data.executable_sections import get_executable_section
from malweave.data.ghidra import lift_sample
from malweave.data.io.archives import get_data_from_archives
from malweave.data.obfuscation import is_obfuscated_bytes
from malweave.data.pe_architecture import get_pe_architecture_from_bytes

RAW_DIR = PROJECT_ROOT / 'data' / 'ranDS' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'ranDS' / 'processed' / 'reproduction'
ARCHIVES = sorted(RAW_DIR.glob('*.zip'))

# None processes every sample. Set a small number for a quick smoke test.
LIMIT = None
# True generates DIS and DEC with Ghidra; this is the slowest stage.
RUN_GHIDRA = True
OVERWRITE = False

assert ARCHIVES, f'No ZIP archives found in {RAW_DIR}'
docker_check = subprocess.run(['docker', 'info'], capture_output=True, text=True)
assert docker_check.returncode == 0, 'The Docker daemon is not running. Start Docker Desktop and run all cells again.'
print('Archives:', [path.name for path in ARCHIVES])
print('Output:', OUTPUT_DIR)

Archives: ['data.zip']
Output: /Users/jung/Projects/malrec-lab/malweave/data/ranDS/processed/reproduction


### 2.1. Compose the existing preprocessing functions into one flow

Preprocessing is organized around five functions: `get_data_from_archives`, `get_pe_architecture_from_bytes`, `is_obfuscated_bytes`, `get_executable_section`, and `lift_sample`. The code below calls them in the paper's preprocessing order, writes accepted samples as actual files, and saves a manifest describing the outcome for every sample. Docker execution is encapsulated inside the DIE and Ghidra functions, so the notebook works entirely through Python APIs.

In [20]:
@dataclass
class PreprocessRecord:
    name: str
    architecture: str = 'unknown'
    obfuscated: bool | None = None
    status: str = 'rejected'
    reason: str = ''
    raw_size: int = 0
    exe_size: int = 0
    raw_path: str | None = None
    exe_path: str | None = None
    dis_path: str | None = None
    dec_path: str | None = None

def preprocess_archives(archives, output_dir, *, limit=None, run_ghidra=False, overwrite=False, verbose=True):
    output_dir = Path(output_dir)
    records = []
    seen_names = set()

    for index, (member_name, content) in enumerate(get_data_from_archives(archives)):
        if limit is not None and index >= limit:
            break
        name = Path(member_name).name
        if not name or name in seen_names:
            raise ValueError(f'Empty or duplicate sample name across archives: {member_name!r}')
        seen_names.add(name)
        record = PreprocessRecord(name=name, raw_size=len(content))
        if verbose:
            print(f'\n[{index + 1}] {name}')
            print(f'  RAW  | {len(content):,} bytes | head: {content[:16].hex(" ")}')

        try:
            record.architecture = get_pe_architecture_from_bytes(content)
            if verbose:
                print(f'  ARCH | {record.architecture}')
            if record.architecture != 'x86':
                record.reason = f'architecture:{record.architecture}'
                if verbose:
                    print(f'  DROP | {record.reason}')
                records.append(record)
                continue

            record.obfuscated = is_obfuscated_bytes(content)
            if verbose:
                print(f'  DIE  | obfuscated={record.obfuscated}')
            if record.obfuscated:
                record.reason = 'obfuscated'
                if verbose:
                    print('  DROP | obfuscated')
                records.append(record)
                continue

            executable = get_executable_section(content=content)
            if executable is None:
                record.reason = 'no-executable-section'
                if verbose:
                    print('  DROP | no-executable-section')
                records.append(record)
                continue

            raw_path = output_dir / 'raw' / name
            exe_path = output_dir / 'exe' / f'{name}.bin'
            raw_path.parent.mkdir(parents=True, exist_ok=True)
            exe_path.parent.mkdir(parents=True, exist_ok=True)
            if overwrite or not raw_path.exists():
                raw_path.write_bytes(content)
            if overwrite or not exe_path.exists():
                exe_path.write_bytes(executable)

            record.status = 'kept'
            record.exe_size = len(executable)
            record.raw_path = str(raw_path)
            record.exe_path = str(exe_path)
            if verbose:
                print(f'  EXE  | {len(executable):,} bytes -> {exe_path.name}')
            if run_ghidra:
                if verbose:
                    print('  GHIDRA | lifting DIS/DEC ...')
                lift_result = lift_sample(raw_path, output_dir, overwrite=overwrite)
                dis_path, dec_path = lift_result.dis_path, lift_result.dec_path
                record.dis_path, record.dec_path = str(dis_path), str(dec_path)
                if verbose:
                    source = 'cache' if lift_result.cached else 'Ghidra'
                    print(f'  KEEP | {source}: DIS={dis_path.stat().st_size:,} bytes, DEC={dec_path.stat().st_size:,} bytes')
            elif verbose:
                print('  KEEP | RAW/EXE created; Ghidra skipped')
        except Exception as error:
            # A malformed sample should not discard the audit trail for the batch.
            record.status = 'error'
            record.reason = f'{type(error).__name__}: {error}'
            if verbose:
                print(f'  ERROR | {record.reason}')
        records.append(record)

    manifest = output_dir / 'manifest.jsonl'
    manifest.parent.mkdir(parents=True, exist_ok=True)
    with manifest.open('w') as stream:
        for record in records:
            stream.write(json.dumps(asdict(record), sort_keys=True) + '\n')
    if verbose:
        print(f'\nMANIFEST | {len(records)} records -> {manifest}')
    return records

def summarize_records(records):
    summary = {'total': len(records), 'kept': 0}
    for record in records:
        key = 'kept' if record.status == 'kept' else f'{record.status}:{record.reason}'
        summary[key] = summary.get(key, 0) + 1
    return summary

## 3. Inspect input without extracting or executing it

This cell only reads each ZIP central directory to count members and inspect their sizes. Binary contents are not read until the pipeline starts.

In [21]:
archive_inventory = []
for archive in ARCHIVES:
    with zipfile.ZipFile(archive) as stream:
        files = [item for item in stream.infolist() if not item.is_dir()]
    archive_inventory.append({
        'archive': archive.name,
        'samples': len(files),
        'uncompressed_MiB': round(sum(item.file_size for item in files) / 2**20, 2),
        'head': [
            {'name': item.filename, 'size': item.file_size}
            for item in files[:5]
        ],
    })
pprint(archive_inventory)

[{'archive': 'data.zip',
  'head': [{'name': 'd0d87cb5e049db6d429e8102c6457ae19a87e3cccfa487b670d4638ea6eaaaa7',
            'size': 168960},
           {'name': 'd0d2525c3cdd04cabdd2148cfbf93b8adc15d14d1b8a2b02d5bbc8eb69f5e0b0',
            'size': 368118},
           {'name': 'd0d8504d8c8baf1973ff40d76e05d7813366afb905be56e9f69184e35da9f58e',
            'size': 973312},
           {'name': 'd02eea14bca5deebe54bb5ad1d865a27d91c3e56f314c1fa5a576b74d4e6a013',
            'size': 1053184},
           {'name': 'd09c89ed44176a65696aa1fb02285e79c4de57a8f359ec5c40e6219908651ecf',
            'size': 12507648}],
  'samples': 11,
  'uncompressed_MiB': 17.39}]


## 4. Run filters and materialize actual data

Each sample is read from its archive once. A sample is written only after satisfying all three conditions: it is PE32 x86, DIE does not flag it as obfuscated, and it contains at least one valid executable/code section.

When `RUN_GHIDRA=True`, only accepted samples are sent to Ghidra. Rerunning with `OVERWRITE=False` reuses existing DIS/DEC files.

In [22]:
records = preprocess_archives(
    ARCHIVES,
    OUTPUT_DIR,
    limit=LIMIT,
    run_ghidra=RUN_GHIDRA,
    overwrite=OVERWRITE,
    verbose=True,
)
pprint(summarize_records(records))
assert any(record.status == 'kept' for record in records), 'No samples passed the filters; inspect the errors in records.'


[1] d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91
  RAW  | 78,336 bytes | head: 4d 5a 90 00 03 00 00 00 04 00 00 3e ff ff 00 00
  ARCH | x86
  DIE  | obfuscated=False
  EXE  | 7,680 bytes -> d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.bin
  GHIDRA | lifting DIS/DEC ...
  KEEP | cache: DIS=37,198 bytes, DEC=39,109 bytes

[2] d02eea14bca5deebe54bb5ad1d865a27d91c3e56f314c1fa5a576b74d4e6a013
  RAW  | 1,053,184 bytes | head: 4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00
  ARCH | x86
  DIE  | obfuscated=True
  DROP | obfuscated

[3] d054e33de2d63966c68b44dd1d1de8a9b7abb76781100fe82423c80e112d4580
  RAW  | 2,083,328 bytes | head: 4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00
  ARCH | x86
  DIE  | obfuscated=True
  DROP | obfuscated

[4] d07092d99a764b583259254d2be9c346c652e747ee58a2e27756a37c42032c48
  RAW  | 290,816 bytes | head: 4d 5a 90 00 03 00 00 00 04 00 00 00 ff ff 00 00
  ARCH | x86
  DIE  | obfuscated=False
  EXE  | 101,888 bytes -> d07092

## 5. Audit filter results

`status=kept` means the sample has materialized `raw` and `EXE` files. If Ghidra is enabled and lifting succeeds, the record also contains `DIS/DEC` paths. Rejected records are not written to the data directories.

In [23]:
for record in records:
    print(
        f'{record.name[:12]}  status={record.status:<8} '
        f'arch={record.architecture:<7} packed={str(record.obfuscated):<5} '
        f'raw={record.raw_size:>9,} exe={record.exe_size:>9,}  {record.reason}'
    )

d00630d78796  status=kept     arch=x86     packed=False raw=   78,336 exe=    7,680  
d02eea14bca5  status=rejected arch=x86     packed=True  raw=1,053,184 exe=        0  obfuscated
d054e33de2d6  status=rejected arch=x86     packed=True  raw=2,083,328 exe=        0  obfuscated
d07092d99a76  status=kept     arch=x86     packed=False raw=  290,816 exe=  101,888  
d079e9fcd6bb  status=kept     arch=x86     packed=False raw=  367,616 exe=  202,240  
d098ebd6d83c  status=kept     arch=x86     packed=False raw=  338,432 exe=   62,976  
d09c89ed4417  status=rejected arch=x64     packed=None  raw=12,507,648 exe=        0  architecture:x64
d09ef86191b7  status=kept     arch=x86     packed=False raw=    4,608 exe=    2,048  
d0d2525c3cdd  status=kept     arch=x86     packed=False raw=  368,118 exe=  237,568  
d0d8504d8c8b  status=kept     arch=x86     packed=False raw=  973,312 exe=  674,304  
d0d87cb5e049  status=kept     arch=x86     packed=False raw=  168,960 exe=   79,872  


## 6. Inspect materialized artifacts

The notebook does not display complete malware contents. It only shows file counts and sizes, short RAW/EXE hexadecimal prefixes, and the first few lines of DIS, DEC, the Ghidra log, and the manifest.

In [24]:
for representation, pattern in [('raw', '*'), ('exe', '*.bin'), ('dis', '*.asm'), ('dec', '*.c')]:
    directory = OUTPUT_DIR / representation
    files = sorted(directory.glob(pattern)) if directory.exists() else []
    print(f'{representation.upper():>3}: {len(files):>3} files, {sum(f.stat().st_size for f in files):>12,} bytes')

kept = next(record for record in records if record.status == 'kept')
raw_path = Path(kept.raw_path)
exe_path = Path(kept.exe_path)
print(f'\nRAW example: {raw_path.name}')
print(raw_path.read_bytes()[:64].hex(' '))
print(f'\nEXE example: {exe_path.name}')
print(exe_path.read_bytes()[:64].hex(' '))

RAW:   8 files,    2,590,198 bytes
EXE:   8 files,    1,368,576 bytes
DIS:   8 files,    1,404,756 bytes
DEC:   8 files,   12,936,213 bytes

RAW example: d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91
4d 5a 90 00 03 00 00 00 04 00 00 3e ff ff 00 00 b8 00 00 00 00 00 00 00 40 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 d8 00 00 00

EXE example: d00630d78796caf768661e92c3f00a404067b033f4f7dce336801ea721ad3a91.bin
55 8b ec 83 ec 14 0b 05 4c 41 41 00 a1 dc 52 41 00 81 15 7c 34 40 00 54 41 41 00 c7 45 f8 dc ef f2 0d c7 45 ec db ef f2 0d c7 45 f4 00 00 00 00 89 45 f4 11 15 4c 10 41 00 a1 cc 52 41 00 81 1d


In [25]:
if kept.dis_path and kept.dec_path:
    dis_lines = Path(kept.dis_path).read_text(errors='replace').splitlines()[:12]
    dec_lines = Path(kept.dec_path).read_text(errors='replace').splitlines()[:12]
    print('DIS example (instruction-only):')
    print('\n'.join(dis_lines))
    print('\nDEC example (C-like):')
    print('\n'.join(dec_lines))
    log_path = OUTPUT_DIR / 'logs' / f'{kept.name}.log'
    if log_path.exists():
        print('\nGhidra log head:')
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[:12]))
else:
    print('DIS/DEC was not generated because RUN_GHIDRA=False or Ghidra failed; inspect manifest.jsonl.')

print('\nManifest head:')
manifest_path = OUTPUT_DIR / 'manifest.jsonl'
print('\n'.join(manifest_path.read_text().splitlines()[:3]))

DIS example (instruction-only):
PUSH EBP
MOV EBP,ESP
SUB ESP,0x14
OR EAX,dword ptr [0x0041414c]
MOV EAX,[0x004152dc]
ADC dword ptr [0x0040347c],0x414154
MOV dword ptr [EBP + -0x8],0xdf2efdc
MOV dword ptr [EBP + -0x14],0xdf2efdb
MOV dword ptr [EBP + -0xc],0x0
MOV dword ptr [EBP + -0xc],EAX
ADC dword ptr [0x0041104c],EDX
MOV EAX,[0x004152cc]

DEC example (C-like):

/* WARNING: Globals starting with '_' overlap smaller symbols at the same address */

void __fastcall FUN_00401000(undefined4 param_1,uint param_2)

{
  undefined *puVar1;
  DWORD DVar2;
  int iVar3;
  uint uVar4;
  uint extraout_ECX;
  uint unaff_EBX;

Ghidra log head:
2026-09-03 16:11:35 INFO  (LoggingInitialization) Using log config file: jar:file:/opt/ghidra/Ghidra/Framework/Generic/lib/Generic.jar!/generic.log4j.xml  
2026-09-03 16:11:35 INFO  (LoggingInitialization) Using log file: /output/log.txt  
2026-09-03 16:11:35 DEBUG (GhidraObjectInputFilter) Including serial input filter: FileSystem/data/client.rmi.serial.filter

## 7. Output contract and next steps

```text
reproduction/
├── manifest.jsonl       # audit trail, not model input
├── raw/<sha256>         # complete PE after filtering
├── exe/<sha256>.bin     # executable-section bytes
├── dis/<sha256>.asm     # one instruction per line
├── dec/<sha256>.c       # decompiled C-like code
└── logs/<sha256>.log    # Ghidra logs for preprocessing diagnostics
```

This notebook ends at the current implementation boundary. The next reproduction stages are representation-level digesting and deduplication, representation-specific BPE pre-tokenization for EXE/DIS/DEC, leakage-safe dataset splitting, CLM/MLM pretraining, and downstream finetuning.